In [ ]:
import os
import json
from PIL import Image
from tqdm import tqdm

# PATHS
OCR_INPUT_DIR = r"D:\Y4 Research\datasets\dietary Images\OCR"
OCR_OUTPUT_DIR = r"D:\Y4 Research\datasets\dietary Images\OCR_cleaned"
IMAGE_DIR = r"D:\Y4 Research\datasets\dietary Images\Set1"

MIN_CONF = 40
os.makedirs(OCR_OUTPUT_DIR, exist_ok=True)

# ---------------------------------------------------
# Build image index (base_name -> full path)
# ---------------------------------------------------
image_index = {}

for img in os.listdir(IMAGE_DIR):
    if img.lower().endswith((".png", ".jpg", ".jpeg")):
        base = os.path.splitext(img)[0]
        image_index[base] = os.path.join(IMAGE_DIR, img)

print(f"✔ Indexed {len(image_index)} images")


# ---------------------------------------------------
# Function to clean OCR & normalize boxes
# ---------------------------------------------------
def prepare_layoutlmv3_input(ocr_json, image_id, image_path, min_conf=40):
    image = Image.open(image_path).convert("RGB")
    W, H = image.size

    tokens, boxes = [], []

    for w in ocr_json.get("words", []):
        if w.get("confidence", 0) < min_conf:
            continue

        x, y, bw, bh = w["bbox"]

        box = [
            int((x / W) * 1000),
            int((y / H) * 1000),
            int(((x + bw) / W) * 1000),
            int(((y + bh) / H) * 1000),
        ]

        tokens.append(w["text"])
        boxes.append(box)

    # Return JSON-safe dict with image_id
    return {
        "image_id": image_id,
        "tokens": tokens,
        "bboxes": boxes
    }


# ---------------------------------------------------
# Batch processing
# ---------------------------------------------------
ocr_files = [f for f in os.listdir(OCR_INPUT_DIR) if f.endswith(".json")]
missing_images = []

for ocr_file in tqdm(ocr_files, desc="Cleaning OCR files"):
    ocr_path = os.path.join(OCR_INPUT_DIR, ocr_file)

    with open(ocr_path, "r", encoding="utf-8") as f:
        ocr_json = json.load(f)

    base_name = os.path.splitext(ocr_file)[0]
    image_path = image_index.get(base_name)

    if image_path is None:
        missing_images.append(base_name)
        continue

    # Pass image_id to the function
    cleaned_data = prepare_layoutlmv3_input(
        ocr_json=ocr_json,
        image_id=base_name,
        image_path=image_path,
        min_conf=MIN_CONF
    )

    output_path = os.path.join(OCR_OUTPUT_DIR, ocr_file)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(cleaned_data, f, indent=2, ensure_ascii=False)

# ---------------------------------------------------
# Report
# ---------------------------------------------------
print(f"\n✔ Completed")
print(f"✔ OCR processed: {len(ocr_files) - len(missing_images)}")
print(f"⚠ Missing images: {len(missing_images)}")


✔ Indexed 100 images


Cleaning OCR files: 100%|██████████| 100/100 [00:00<00:00, 124.75it/s]


✔ Completed
✔ OCR processed: 100
⚠ Missing images: 0


In [5]:
from paddleocr import PaddleOCR

ocr = PaddleOCR(
    lang="en",
    use_textline_orientation=True  # keep False since CUDA not installed
)

c:\Projects\Research_Project\env\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\rashm\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\rashm\.paddlex\official_models\UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\rashm\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None)
Mod

In [6]:
from paddleocr import PaddleOCR
import json
from PIL import Image

def extract_ocr_data(image_path, image_id):
    """
    Extract OCR data from an image and format it as tokens and bounding boxes.
    
    Args:
        image_path: Path to the image file
        image_id: Identifier for the image
    
    Returns:
        Dictionary with image_id, tokens, and bboxes
    """
    # Initialize PaddleOCR
    ocr = PaddleOCR(use_textline_orientation=True, lang='en')
    
    # Perform OCR
    result = ocr.predict(image_path)
    
    # Initialize output structure
    output = {
        "image_id": image_id,
        "tokens": [],
        "bboxes": []
    }
    
    # Debug: print the result structure to understand the format
    print("Result type:", type(result))
    if isinstance(result, dict):
        print("Dict keys:", result.keys())
        if result:
            print("Sample keys and types:", {k: type(v) for k, v in list(result.items())[:3]})
    elif isinstance(result, list) and len(result) > 0:
        print("List length:", len(result))
        print("First item type:", type(result[0]))
        if isinstance(result[0], list) and len(result[0]) > 0:
            print("Sample item:", result[0][0])
    
    # Process OCR results - handle new PaddleOCR format
    if result:
        # Check if result is a dict (new format) or list (old format)
        if isinstance(result, dict):
            # New format: result is a dict with 'dt_polys' and 'rec_text'
            if 'dt_polys' in result and 'rec_text' in result:
                bboxes = result['dt_polys']
                texts = result['rec_text']
                
                for bbox, text_info in zip(bboxes, texts):
                    # Extract text (might be a tuple or string)
                    text = text_info[0] if isinstance(text_info, tuple) else text_info
                    
                    # Convert bbox to [x1, y1, x2, y2]
                    if len(bbox) == 4 and len(bbox[0]) == 2:  # [[x1,y1],[x2,y2],[x3,y3],[x4,y4]]
                        x_coords = [point[0] for point in bbox]
                        y_coords = [point[1] for point in bbox]
                    elif len(bbox) == 4:  # [x1, y1, x2, y2]
                        x1, y1, x2, y2 = bbox
                        x_coords = [x1, x2]
                        y_coords = [y1, y2]
                    else:
                        continue
                    
                    x1 = int(min(x_coords))
                    y1 = int(min(y_coords))
                    x2 = int(max(x_coords))
                    y2 = int(max(y_coords))
                    
                    # Split text into tokens
                    words = text.split()
                    
                    if len(words) > 1:
                        word_width = (x2 - x1) / len(words)
                        for i, word in enumerate(words):
                            word_x1 = int(x1 + i * word_width)
                            word_x2 = int(x1 + (i + 1) * word_width)
                            
                            output["tokens"].append(word)
                            output["bboxes"].append([word_x1, y1, word_x2, y2])
                    else:
                        output["tokens"].append(text)
                        output["bboxes"].append([x1, y1, x2, y2])
        
        # Old format: result is a list
        elif isinstance(result, list) and len(result) > 0 and result[0]:
            for line in result[0]:
                print(f"Processing line: {line}")  # Debug
                print(f"Line type: {type(line)}, Length: {len(line) if hasattr(line, '__len__') else 'N/A'}")  # Debug
                
                # Extract bounding box coordinates and text
                bbox = line[0]
                text = line[1]
                
                print(f"Bbox: {bbox}, type: {type(bbox)}")  # Debug
                print(f"Text: {text}, type: {type(text)}")  # Debug
                
                # Handle both old format (text, confidence) and new format (just text)
                if isinstance(text, tuple):
                    text, confidence = text
                
                # Convert bbox - handle different formats
                if isinstance(bbox, (list, tuple)) and len(bbox) > 0:
                    # Check if it's [[x1,y1],[x2,y2],...] format
                    if isinstance(bbox[0], (list, tuple)) and len(bbox[0]) == 2:
                        x_coords = [point[0] for point in bbox]
                        y_coords = [point[1] for point in bbox]
                    # Check if it's [x1, y1, x2, y2] format
                    elif len(bbox) == 4 and all(isinstance(x, (int, float)) for x in bbox):
                        x_coords = [bbox[0], bbox[2]]
                        y_coords = [bbox[1], bbox[3]]
                    else:
                        print(f"Unknown bbox format: {bbox}")
                        continue
                else:
                    print(f"Invalid bbox: {bbox}")
                    continue
                
                x1 = int(min(x_coords))
                y1 = int(min(y_coords))
                x2 = int(max(x_coords))
                y2 = int(max(y_coords))
                
                # Split text into tokens (words)
                words = text.split()
                
                # For multi-word lines, distribute bbox proportionally
                if len(words) > 1:
                    word_width = (x2 - x1) / len(words)
                    for i, word in enumerate(words):
                        word_x1 = int(x1 + i * word_width)
                        word_x2 = int(x1 + (i + 1) * word_width)
                        
                        output["tokens"].append(word)
                        output["bboxes"].append([word_x1, y1, word_x2, y2])
                else:
                    # Single word
                    output["tokens"].append(text)
                    output["bboxes"].append([x1, y1, x2, y2])
    
    return output

# Example usage
if __name__ == "__main__":
    # Process your vitamin B2 label image
    image_path = r"E:\Ex\542.png"
    image_id = "542 conv 0"
    
    # Extract OCR data
    result = extract_ocr_data(image_path, image_id)
    
    # Print formatted JSON
    print(json.dumps(result, indent=2))
    
    # Optionally save to file
    with open(r"E:\sample_annotation.json", "w") as f:
        json.dump(result, f, indent=2)
    
    print(f"\nExtracted {len(result['tokens'])} tokens")

Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\rashm\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\rashm\.paddlex\official_models\UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\rashm\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\rashm\.paddlex\official_models\PP-OCRv5_server_det`.
Creating model: ('en_PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\rashm\.paddlex\

Result type: <class 'list'>
List length: 1
First item type: <class 'paddlex.inference.pipelines.ocr.result.OCRResult'>
Processing line: input_path
Line type: <class 'str'>, Length: 10
Bbox: i, type: <class 'str'>
Text: n, type: <class 'str'>
Invalid bbox: i
Processing line: page_index
Line type: <class 'str'>, Length: 10
Bbox: p, type: <class 'str'>
Text: a, type: <class 'str'>
Invalid bbox: p
Processing line: doc_preprocessor_res
Line type: <class 'str'>, Length: 20
Bbox: d, type: <class 'str'>
Text: o, type: <class 'str'>
Invalid bbox: d
Processing line: dt_polys
Line type: <class 'str'>, Length: 8
Bbox: d, type: <class 'str'>
Text: t, type: <class 'str'>
Invalid bbox: d
Processing line: model_settings
Line type: <class 'str'>, Length: 14
Bbox: m, type: <class 'str'>
Text: o, type: <class 'str'>
Invalid bbox: m
Processing line: text_det_params
Line type: <class 'str'>, Length: 15
Bbox: t, type: <class 'str'>
Text: e, type: <class 'str'>
Invalid bbox: t
Processing line: text_type
Line

In [7]:
import pytesseract

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

In [8]:
import cv2
import os

# Define paths
input_path = r"E:\Ex\542.png"
output_path = r"E:\542_preprocessed.png"

def preprocess_for_tesseract(image_path, save_path):
    # 1. Load the image
    img = cv2.imread(image_path)
    
    if img is None:
        print("Error: Could not find or open the image.")
        return

    # 2. Convert to Grayscale (Luminosity Standard)
    # This uses the 0.299R + 0.587G + 0.114B formula automatically
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 3. Apply Otsu's Thresholding
    # This converts the grayscale image into a pure Black & White (Binary) image
    # It automatically calculates the best threshold value for the background
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # 4. Save the processed image
    cv2.imwrite(save_path, thresh)
    print(f"Success! Preprocessed image saved to: {save_path}")

# Run the function
preprocess_for_tesseract(input_path, output_path)

Success! Preprocessed image saved to: E:\542_preprocessed.png


In [11]:
import json
import os
import pytesseract
import cv2

# ==============================
# CONFIGURATION
# ==============================
IMAGE_PATH = r"E:\542_preprocessed.png"       # input image path
OUTPUT_DIR = r"E:\OCR"                      # output folder
OUTPUT_JSON = "sample_annotation.json"
IMAGE_ID = "vitamin_b2_label_001"

#
# ==============================
# CREATE OUTPUT FOLDER
# ==============================
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==============================
# LOAD IMAGE
# ==============================
image = cv2.imread(IMAGE_PATH)
if image is None:
    raise FileNotFoundError(f"Image not found: {IMAGE_PATH}")

# ==============================
# RUN TESSERACT OCR
# ==============================
data = pytesseract.image_to_data(
    image,
    output_type=pytesseract.Output.DICT,
    config="--psm 6"
)

tokens = []
bboxes = []

# ==============================
# PARSE OCR RESULTS
# ==============================
n = len(data["text"])
for i in range(n):
    text = data["text"][i].strip()

    # Skip empty text
    if text == "":
        continue

    x = int(data["left"][i])
    y = int(data["top"][i])
    w = int(data["width"][i])
    h = int(data["height"][i])

    tokens.append(text)
    bboxes.append([x, y, x + w, y + h])

# ==============================
# BUILD JSON STRUCTURE
# ==============================
annotation = {
    "image_id": IMAGE_ID,
    "tokens": tokens,
    "bboxes": bboxes
}

# ==============================
# SAVE JSON FILE
# ==============================
output_path = os.path.join(OUTPUT_DIR, OUTPUT_JSON)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(annotation, f, indent=2, ensure_ascii=False)

print(f"✅ OCR annotation saved at: {output_path}")


✅ OCR annotation saved at: E:\OCR\sample_annotation.json


batch

In [1]:
import cv2
import os

# ==============================
# CONFIGURATION
# ==============================
INPUT_DIR = r"D:\Y4 Research\datasets\dietary Images\Set1"          # folder containing images
OUTPUT_DIR = r"D:\Y4 Research\datasets\dietary Images\Grayscale"  # folder to save processed images

# Supported image extensions
VALID_EXTS = (".png")

# ==============================
# CREATE OUTPUT FOLDER
# ==============================
os.makedirs(OUTPUT_DIR, exist_ok=True)

def preprocess_for_tesseract(input_dir, output_dir):
    for filename in os.listdir(input_dir):

        if not filename.lower().endswith(VALID_EXTS):
            continue

        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, filename)

        # 1. Load image
        img = cv2.imread(input_path)
        if img is None:
            print(f"❌ Skipped (cannot read): {filename}")
            continue

        # 2. Convert to grayscale
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # 3. Otsu thresholding
        _, thresh = cv2.threshold(
            gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )

        # 4. Save processed image
        cv2.imwrite(output_path, thresh)
        print(f"✅ Processed: {filename}")

# ==============================
# RUN
# ==============================
preprocess_for_tesseract(INPUT_DIR, OUTPUT_DIR)


✅ Processed: 1000.png
✅ Processed: 1001.png
✅ Processed: 1002.png
✅ Processed: 1003.png
✅ Processed: 1004.png
✅ Processed: 1005.png
✅ Processed: 1006.png
✅ Processed: 1007.png
✅ Processed: 1008.png
✅ Processed: 1009.png
✅ Processed: 1010.png
✅ Processed: 1011.png
✅ Processed: 1012.png
✅ Processed: 1013.png
✅ Processed: 1014.png
✅ Processed: 1015.png
✅ Processed: 1016.png
✅ Processed: 1017.png
✅ Processed: 1018.png
✅ Processed: 1019.png
✅ Processed: 1020.png
✅ Processed: 1021.png
✅ Processed: 1022.png
✅ Processed: 1023.png
✅ Processed: 1024.png
✅ Processed: 1025.png
✅ Processed: 1026.png
✅ Processed: 1027.png
✅ Processed: 1028.png
✅ Processed: 1029.png
✅ Processed: 1030.png
✅ Processed: 1031.png
✅ Processed: 1032.png
✅ Processed: 1033.png
✅ Processed: 1034.png
✅ Processed: 1035.png
✅ Processed: 1036.png
✅ Processed: 1037.png
✅ Processed: 542.png
✅ Processed: 543.png
✅ Processed: 544.png
✅ Processed: 545.png
✅ Processed: 546.png
✅ Processed: 547.png
✅ Processed: 548.png
✅ Processed: 549.

In [13]:
import cv2
import os

# ==============================
# CONFIGURATION
# ==============================
INPUT_DIR = r"D:\Y4 Research\datasets\dietary Images\Grayscale"
OUTPUT_DIR = r"D:\Y4 Research\datasets\dietary Images\denoised"

VALID_EXTS = (".png")

# ==============================
# CREATE OUTPUT FOLDER
# ==============================
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==============================
# BATCH MEDIAN BLUR
# ==============================
for filename in os.listdir(INPUT_DIR):

    if not filename.lower().endswith(VALID_EXTS):
        continue

    input_path = os.path.join(INPUT_DIR, filename)
    output_path = os.path.join(OUTPUT_DIR, filename)

    # Load grayscale image
    img = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"❌ Skipped (cannot read): {filename}")
        continue

    # Apply 3x3 median blur
    denoised = cv2.medianBlur(img, 3)

    # Save image
    cv2.imwrite(output_path, denoised)
    print(f"✅ Denoised: {filename}")

print("🎉 All images processed successfully!")


✅ Denoised: 1000.png
✅ Denoised: 1001.png
✅ Denoised: 1002.png
✅ Denoised: 1003.png
✅ Denoised: 1004.png
✅ Denoised: 1005.png
✅ Denoised: 1006.png
✅ Denoised: 1007.png
✅ Denoised: 1008.png
✅ Denoised: 1009.png
✅ Denoised: 1010.png
✅ Denoised: 1011.png
✅ Denoised: 1012.png
✅ Denoised: 1013.png
✅ Denoised: 1014.png
✅ Denoised: 1015.png
✅ Denoised: 1016.png
✅ Denoised: 1017.png
✅ Denoised: 1018.png
✅ Denoised: 1019.png
✅ Denoised: 1020.png
✅ Denoised: 1021.png
✅ Denoised: 1022.png
✅ Denoised: 1023.png
✅ Denoised: 1024.png
✅ Denoised: 1025.png
✅ Denoised: 1026.png
✅ Denoised: 1027.png
✅ Denoised: 1028.png
✅ Denoised: 1029.png
✅ Denoised: 1030.png
✅ Denoised: 1031.png
✅ Denoised: 1032.png
✅ Denoised: 1033.png
✅ Denoised: 1034.png
✅ Denoised: 1035.png
✅ Denoised: 1036.png
✅ Denoised: 1037.png
✅ Denoised: 542.png
✅ Denoised: 543.png
✅ Denoised: 544.png
✅ Denoised: 545.png
✅ Denoised: 546.png
✅ Denoised: 547.png
✅ Denoised: 548.png
✅ Denoised: 549.png
✅ Denoised: 550.png
✅ Denoised: 551.png
✅ 

In [2]:
import json
import os
import pytesseract
import cv2

# ==============================
# CONFIGURATION
# ==============================
INPUT_DIR = r"D:\Y4 Research\datasets\dietary Images\Grayscale"
OUTPUT_DIR = r"D:\Y4 Research\datasets\dietary Images\OCR"

VALID_EXTS = (".png", ".jpg", ".jpeg", ".tif", ".tiff")

# ==============================
# CREATE OUTPUT FOLDER
# ==============================
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==============================
# BATCH OCR
# ==============================
for filename in os.listdir(INPUT_DIR):

    if not filename.lower().endswith(VALID_EXTS):
        continue

    image_path = os.path.join(INPUT_DIR, filename)

    image = cv2.imread(image_path)
    if image is None:
        print(f"❌ Skipped (cannot read): {filename}")
        continue

    image_name = os.path.splitext(filename)[0]
    output_json = f"{image_name}.json"
    output_path = os.path.join(OUTPUT_DIR, output_json)

    # ==============================
    # RUN TESSERACT OCR
    # ==============================
    data = pytesseract.image_to_data(
        image,
        output_type=pytesseract.Output.DICT,
        config="--psm 6"
    )

    tokens = []
    bboxes = []
    confidences = []

    for i in range(len(data["text"])):
        text = data["text"][i].strip()
        conf = data["conf"][i]

        # Skip empty text or invalid confidence
        if text == "" or conf == "-1":
            continue

        x = int(data["left"][i])
        y = int(data["top"][i])
        w = int(data["width"][i])
        h = int(data["height"][i])

        tokens.append(text)
        bboxes.append([x, y, x + w, y + h])
        confidences.append(float(conf))

    # ==============================
    # BUILD JSON STRUCTURE
    # ==============================
    annotation = {
        "image_id": filename,
        "tokens": tokens,
        "bboxes": bboxes,
        "confidences": confidences
    }

    # ==============================
    # SAVE JSON
    # ==============================
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(annotation, f, indent=2, ensure_ascii=False)

    print(f"✅ OCR saved: {output_json}")

print("🎉 Batch OCR completed successfully!")


✅ OCR saved: 1000.json
✅ OCR saved: 1001.json
✅ OCR saved: 1002.json
✅ OCR saved: 1003.json
✅ OCR saved: 1004.json
✅ OCR saved: 1005.json
✅ OCR saved: 1006.json
✅ OCR saved: 1007.json
✅ OCR saved: 1008.json
✅ OCR saved: 1009.json
✅ OCR saved: 1010.json
✅ OCR saved: 1011.json
✅ OCR saved: 1012.json
✅ OCR saved: 1013.json
✅ OCR saved: 1014.json
✅ OCR saved: 1015.json
✅ OCR saved: 1016.json
✅ OCR saved: 1017.json
✅ OCR saved: 1018.json
✅ OCR saved: 1019.json
✅ OCR saved: 1020.json
✅ OCR saved: 1021.json
✅ OCR saved: 1022.json
✅ OCR saved: 1023.json
✅ OCR saved: 1024.json
✅ OCR saved: 1025.json
✅ OCR saved: 1026.json
✅ OCR saved: 1027.json
✅ OCR saved: 1028.json
✅ OCR saved: 1029.json
✅ OCR saved: 1030.json
✅ OCR saved: 1031.json
✅ OCR saved: 1032.json
✅ OCR saved: 1033.json
✅ OCR saved: 1034.json
✅ OCR saved: 1035.json
✅ OCR saved: 1036.json
✅ OCR saved: 1037.json
✅ OCR saved: 542.json
✅ OCR saved: 543.json
✅ OCR saved: 544.json
✅ OCR saved: 545.json
✅ OCR saved: 546.json
✅ OCR saved: 547

In [3]:
import os
import json
from PIL import Image
from tqdm import tqdm

# ==============================
# PATHS
# ==============================
OCR_INPUT_DIR = r"D:\Y4 Research\datasets\dietary Images\OCR"
OCR_OUTPUT_DIR = r"D:\Y4 Research\datasets\dietary Images\cleaned_OCR"
IMAGE_DIR = r"D:\Y4 Research\datasets\dietary Images\Set1"

MIN_CONF = 40
os.makedirs(OCR_OUTPUT_DIR, exist_ok=True)

# ==============================
# INDEX IMAGES (name → path)
# ==============================
image_index = {}
for img in os.listdir(IMAGE_DIR):
    if img.lower().endswith((".png", ".jpg", ".jpeg")):
        base = os.path.splitext(img)[0]
        image_index[base] = os.path.join(IMAGE_DIR, img)

print(f"✔ Indexed {len(image_index)} images")

# ==============================
# CLEAN + NORMALIZE FUNCTION
# ==============================
def clean_and_normalize(ocr_json, image_path, image_id, min_conf=40):
    image = Image.open(image_path).convert("RGB")
    W, H = image.size

    tokens, boxes = [], []

    for token, box, conf in zip(
        ocr_json["tokens"],
        ocr_json["bboxes"],
        ocr_json["confidences"]
    ):
        if conf < min_conf:
            continue

        x1, y1, x2, y2 = box

        # Normalize to 0–1000 (LayoutLM standard)
        norm_box = [
            int((x1 / W) * 1000),
            int((y1 / H) * 1000),
            int((x2 / W) * 1000),
            int((y2 / H) * 1000),
        ]

        tokens.append(token)
        boxes.append(norm_box)

    return {
        "image_id": image_id,
        "tokens": tokens,
        "bboxes": boxes
    }

# ==============================
# BATCH PROCESSING
# ==============================
ocr_files = [f for f in os.listdir(OCR_INPUT_DIR) if f.endswith(".json")]
missing_images = []

for ocr_file in tqdm(ocr_files, desc="Cleaning OCR"):
    ocr_path = os.path.join(OCR_INPUT_DIR, ocr_file)

    with open(ocr_path, "r", encoding="utf-8") as f:
        ocr_json = json.load(f)

    base_name = os.path.splitext(ocr_file)[0]
    image_path = image_index.get(base_name)

    if image_path is None:
        missing_images.append(base_name)
        continue

    cleaned = clean_and_normalize(
        ocr_json=ocr_json,
        image_path=image_path,
        image_id=base_name,
        min_conf=MIN_CONF
    )

    output_path = os.path.join(OCR_OUTPUT_DIR, ocr_file)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(cleaned, f, indent=2, ensure_ascii=False)

# ==============================
# REPORT
# ==============================
print("\n✔ Completed")
print(f"✔ OCR files processed: {len(ocr_files) - len(missing_images)}")
print(f"⚠ Missing images: {len(missing_images)}")


✔ Indexed 440 images


Cleaning OCR: 100%|██████████| 440/440 [00:12<00:00, 35.30it/s]


✔ Completed
✔ OCR files processed: 440
⚠ Missing images: 0


In [2]:
import json
from labels import LABELS

In [3]:
import os
import json
import re
import sys
from tqdm import tqdm

# ==============================
# PATHS
# ==============================
OCR_INPUT_DIR = r"D:\Y4 Research\datasets\dietary Images\cleaned_OCR"
OCR_OUTPUT_DIR = r"D:\Y4 Research\datasets\dietary Images\annotated_OCR"
LABELS_DIR = r"C:\Projects\Research_Project\notebooks"  # folder containing labels.py

os.makedirs(OCR_OUTPUT_DIR, exist_ok=True)

# ==============================
# LOAD LABELS (OPTION A)
# ==============================
sys.path.append(LABELS_DIR)
from labels import LABELS   # labels.py must define LABELS = [...]

label2id = {label: idx for idx, label in enumerate(LABELS)}
id2label = {idx: label for label, idx in label2id.items()}

# ==============================
# KEYWORD RULES
# ==============================
INGREDIENT_START = {"ingredients", "ingredients:", "ingredient", "ingredient:"}
INGREDIENT_STOP = {"store", "warning", "supplement", "facts", "serving"}

NUTRIENT_NAMES = {
    # Calories & macros
    "calories",
    "total", "fat",
    "cholesterol",
    "carbohydrates",
    "sugars", "sugar",
    "protein",

    # Vitamins
    "vitamin", "a",
    "c",
    "d",
    "d-3",
    "e",
    "k",

    "thiamin",
    "thiamine",
    "b1",
    "riboflavin",
    "b2",
    "niacin",
    "b3",
    "b6",
    "pyridoxine",
    "folic", "acid",
    "folate",
    "b12",
    "cyanocobalamin",
    "biotin",
    "pantothenic", "acid",

    # Minerals
    "calcium",
    "iron",
    "magnesium",
    "zinc",
    "selenium",
    "copper",
    "manganese",
    "chromium",
    "iodine",
    "molybdenum",
    "boron",

    # Fatty acids
    "omega-3",
    "omega-6",
    "fatty", "acids",
    "epa",
    "dha",
    "gla",

    # Amino acids & compounds
    "l-arginine",
    "arginine",
    "nadh",
    "coenzyme",
    "alpha", "lipoic", "acid",
    "inositol",
    "choline",
    "bitartrate",
    "soy", "lecithin",

    # Enzymes
    "amylase",
    "protease",
    "lipase",
    "cellulase",

    # Phytochemicals
    "green", "tea", "leaf", "extract",
    "citrus", "bioflavonoids", "complex",
    "hesperidin", "complex",
    "quercetin",
    "rutin",
    "lutein",
    "lycopene",
    "tocotrienol", "complex",

    # Herbal ingredients
    "echinacea", "purpurea",
    "herb", "powder",
    "cascara", "sagrada",
    "senna", "extract",
    "sennosides",
    "fennel", "seed",
    "licorice", "root",
    "uva", "ursi", "leaves",

    # Blends
    "herbal", "proprietary", "blend",

    # Oils
    "fish", "oil",
    "body",
    "borage",

    # Chemical forms / sources
    "beta-carotene",
    "retinyl", "acetate",
    "cholecalciferol",
    "ascorbic", "acid",
    "carbonate",
    "ferrous", "sulfate",
    "fumarate",
    "bis-glycinate",
    "oxide",
    "picolinate",
    "hydrolyzed", "protein", "chelate",
    "kelp",
    "l-selenomethionine",
    "cupric", "oxide",
    "sulfate",
    "hydrochloride",
    "mononitrate"

}

CLAIM_WORDS = {
    # Supplement types & dosage forms
    "dietary", "supplement",
    "vitamin",
    "antioxidant",
    "herbal",
    "vegetarian",
    "suitable",
    "for",
    "vegetarians",
    "rapid", "release",
    "timed", "release",
    "enteric", "coated",
    "easy", "to", "swallow",
    "coated", "caplets",
    "softgels",
    "capsules",
    "tablets",

    # Supports / promotes (health claims)
    "supports", "immune", "system",
    "health",
    "heart",
    "cardiovascular",
    "coronary",
    "brain",
    "eye",
    "skin",
    "joint",
    "digestive",
    "digestion",
    "energy",
    "production",
    "metabolism",
    "fat",
    "mental",
    "alertness",
    "nervous",

    "promotes",
    "antioxidant",
    "protection",
    "support",
    "regularity",
    "circulation",

    # Helps / may reduce
    "helps",
    "maintain",
    "normal",
    "cholesterol",
    "levels",
    "blood",
    "pressure",
    "reduce",
    "risk",
    "heart", "disease",
    "may",

    # Blood / oxygen related
    "red",
    "blood",
    "cell",
    "cells",
    "production",
    "formation",
    "essential",
    "involved",
    "plays",
    "role",
    "oxygen",
    "transport",
    "utilization",

    # Antioxidant language
    "powerful",
    "protects",
    "against",
    "free",
    "radicals",
    "protects",
    "cells",
    "oxidative",
    "damage",

    # Immune wording
    "immune",
    "response",
    "natural",
    "resistance",
    "defense",

    # Digestive / laxative
    "occasional",
    "constipation",
    "gentle",
    "yet",
    "effective",
    "cleansing",
    "ease",
    "laxative",
    "all",
    "natural",
    "herbs",

    # Absorption / enzymes
    "enzymes",
    "nutrient",
    "absorption",

    # Mental / cognitive
    "focus",
    "cognitive",
    "function",

    # Potency / strength
    "high",
    "potency",
    "extra",
    "strength",
    "double",
    "maximum",

    # Quality / testing
    "clinically",
    "tested",
    "scientifically",
    "designed",
    "laboratory",
    "quality",
    "assured",
    "verified",
    "rigorously",
    "pharmaceutical",
    "grade",

    # Bioavailability / form
    "purified",
    "highly",
    "bioavailable",
    "superior",
    "fast",
    "acting",
    "free",
    "form",

    # Free-from claims
    "no",
    "artificial",
    "color",
    "flavor",
    "sweetener",
    "preservatives",
    "sugar",
    "starch",
    "milk",
    "lactose",
    "soy",
    "gluten",
    "wheat",
    "yeast",
    "fish",
    "sodium",

    # Digestive comfort
    "stomach",
    "easy",
    "on",
    "digestion",
    "non",
    "irritating",

    # General wellness
    "overall",
    "general",
    "wellness",
    "dense",
    "formula",
    "complete",

    # Legal / FDA disclaimer
    "these",
    "statements",
    "have",
    "not",
    "been",
    "evaluated",
    "by",
    "the",
    "food",
    "and",
    "drug",
    "administration",
    "this",
    "product",
    "is",
    "intended",
    "to",
    "diagnose",
    "treat",
    "cure",
    "or",
    "prevent",
    "any",
    "disease"

}

INGREDIENT_WORDS = [
    "vitamin","a","beta","carotene","retinol","retinyl","acetate","palmitate",
    "vitamin","c","ascorbic","acid","calcium","ascorbate","sodium","ascorbate",
    "vitamin","d","d2","d3","ergocalciferol","cholecalciferol",
    "vitamin","e","tocopherol","alpha","beta","gamma","delta","succinate",
    "vitamin","k","k1","k2","phylloquinone","menaquinone",
    "thiamin","thiamine","vitamin","b1","mononitrate",
    "riboflavin","vitamin","b2",
    "niacin","niacinamide","nicotinic","acid","vitamin","b3",
    "pantothenic","acid","pantothenate","vitamin","b5",
    "pyridoxine","hydrochloride","vitamin","b6",
    "biotin","vitamin","b7",
    "folic","acid","folate","vitamin","b9",
    "cyanocobalamin","methylcobalamin","vitamin","b12",
    "choline","bitartrate","inositol","paba",
    "calcium","carbonate","citrate","phosphate","dicalcium","tricalcium",
    "iron","ferrous","fumarate","sulfate","bisglycinate","glycinate",
    "magnesium","oxide","citrate","stearate",
    "zinc","oxide","gluconate","picolinate","chelate",
    "copper","oxide","gluconate","chelate",
    "manganese","sulfate","gluconate",
    "selenium","selenomethionine","sodium","selenite",
    "chromium","picolinate","nicotinate","chelate",
    "molybdenum","molybdate",
    "iodine","potassium","iodide","kelp",
    "boron","borate","silicon","silica",
    "potassium","chloride","phosphate",
    "omega","3","omega","6","epa","dha","ala",
    "fish","oil","fish-body-oil","anchovy","sardine",
    "borage","oil","evening","primrose","oil",
    "flaxseed","oil","sunflower","oil","soybean","oil",
    "coenzyme","q10","ubiquinone","ubiquinol","q-sorb",
    "nadh","nicotinamide","adenine","dinucleotide",
    "l-carnitine","tartrate","free","form",
    "l-arginine","l-lysine","l-glutamine","taurine",
    "protein","hydrolysate","amino","acids",
    "citrus","bioflavonoids","bioflavonoid","complex",
    "hesperidin","rutin","quercetin","lutein","lycopene",
    "tocotrienols",
    "green","tea","extract","camellia","sinensis",
    "echinacea","purpurea","extract","herb","powder",
    "senna","cassia","angustifolia","extract",
    "cascara","sagrada","rhamnus","purshiana",
    "fennel","seed","licorice","root","uva","ursi","leaf",
    "rose","hips","wild","rose","hip",
    "cranberry","concentrate","powder",
    "parsley","alfalfa","watercress",
    "gelatin","vegetable","cellulose","hypromellose",
    "croscarmellose","crospovidone",
    "stearic","acid","magnesium","stearate",
    "silica","titanium","dioxide",
    "beeswax","yellow","white",
    "glycerin","vegetable","glycerin",
    "sorbitol","mannitol","xylitol",
    "sucrose","fructose","glucose",
    "maltodextrin","corn","starch","rice","flour","rice","bran",
    "lecithin","soy","lecithin",
    "natural","flavor","lemon","orange","vanilla",
    "citric","acid","malic","acid",
    "potassium","sorbate","sodium","benzoate",
    "caramel","color","annatto","carmine",
    "chlorophyll","chlorophyllin",
    "polysorbate","80",
    "medium","chain","triglycerides",
    "shellac","food","glaze",
    "purified","water",
    "calories","fat","total","cholesterol","carbohydrate","sugars"
]


UNITS = {"mg", "g", "mcg", "%", "iu", "kcal"}

# ==============================
# HELPERS
# ==============================
def norm(t: str) -> str:
    return t.lower().strip()

def is_number(t: str) -> bool:
    return bool(re.fullmatch(r"[0-9,.]+", t))

# ==============================
# AUTO-LABEL FUNCTION (BIOES-SAFE)
# ==============================
def auto_label(tokens):
    labels = ["O"] * len(tokens)
    i = 0

    while i < len(tokens):
        t = norm(tokens[i])

        # ---------- INGREDIENT HEADER ----------
        if t in INGREDIENT_START:
            labels[i] = "O"
            i += 1
            continue

        # ---------- INGREDIENT SPAN ----------
        if t in INGREDIENT_WORDS:
            start = i
            j = i + 1
            while j < len(tokens) and norm(tokens[j]) in INGREDIENT_WORDS:
                j += 1

            if j - start == 1:
                labels[start] = "S-INGREDIENT"
            else:
                labels[start] = "B-INGREDIENT"
                for k in range(start + 1, j - 1):
                    labels[k] = "I-INGREDIENT"
                labels[j - 1] = "E-INGREDIENT"

            i = j
            continue

        # ---------- NUTRIENT NAME ----------
        if t in NUTRIENT_NAMES:
            labels[i] = "S-NUTRIENT_NAME"
            i += 1
            continue

        # ---------- NUTRIENT VALUE ----------
        if is_number(t) and i + 1 < len(tokens):
            if norm(tokens[i + 1]) in UNITS:
                labels[i] = "B-NUTRIENT_VALUE"
                labels[i + 1] = "E-NUTRIENT_VALUE"
                i += 2
                continue

        if "%" in t:
            labels[i] = "S-NUTRIENT_VALUE"
            i += 1
            continue

        # ---------- CLAIM SPAN ----------
        if t in CLAIM_WORDS:
            start = i
            j = i + 1
            while j < len(tokens) and norm(tokens[j]) in CLAIM_WORDS:
                j += 1

            if j - start == 1:
                labels[start] = "S-CLAIMS"
            else:
                labels[start] = "B-CLAIMS"
                for k in range(start + 1, j - 1):
                    labels[k] = "I-CLAIMS"
                labels[j - 1] = "E-CLAIMS"

            i = j
            continue

        i += 1

    return labels

# ==============================
# BATCH PROCESS + SAVE
# ==============================
ocr_files = [f for f in os.listdir(OCR_INPUT_DIR) if f.endswith(".json")]

for file in tqdm(ocr_files, desc="Auto annotating"):
    in_path = os.path.join(OCR_INPUT_DIR, file)
    out_path = os.path.join(OCR_OUTPUT_DIR, file)

    with open(in_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    tokens = data.get("tokens", [])
    bboxes = data.get("bboxes", [])

    # Always generate labels (even if empty)
    labels = auto_label(tokens) if tokens else []

    output = {
        "image_id": data.get("image_id", file),
        "tokens": tokens,
        "bboxes": bboxes,
        "labels": labels
    }

    # ✅ SAVE annotated JSON
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)

print("✔ Auto annotation completed and saved to output folder")

Auto annotating: 100%|██████████| 440/440 [00:01<00:00, 352.88it/s]

✔ Auto annotation completed and saved to output folder


In [4]:

import json
from pathlib import Path

# Path to annotated JSONs
ANNOTATED_DIR = Path(r"D:\Y4 Research\datasets\dietary Images\annotated_OCR")

# Loop through all JSON files
for jf in ANNOTATED_DIR.glob("*.json"):
    with open(jf, encoding="utf-8") as f:
        data = json.load(f)

    # Check length consistency
    assert len(data["tokens"]) == len(data["bboxes"]) == len(data["labels"]), \
        f"{jf.name}: tokens/bboxes/labels length mismatch"

    # Check label validity
    for l in data["labels"]:
        assert l in LABELS, f"{jf.name}: invalid label '{l}'"

print("✅ All annotations valid")


✅ All annotations valid


In [5]:
from labels import LABELS

label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

In [6]:
import json
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    LayoutLMv3Processor,
    LayoutLMv3ForTokenClassification
)

In [7]:
ANN_DIR = r"D:\Y4 Research\datasets\dietary Images\For training\OCR"
IMG_DIR = r"D:\Y4 Research\datasets\dietary Images\For training\Images"

In [8]:
processor = LayoutLMv3Processor.from_pretrained(
    "microsoft/layoutlmv3-base",
    apply_ocr=False
)

In [10]:
from labels import label2id

class FoodLabelDataset(Dataset):
    def __init__(self, ann_dir, img_dir, processor):
        self.files = list(Path(ann_dir).glob("*.json"))
        self.img_dir = Path(img_dir)
        self.processor = processor

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        ann = json.load(open(self.files[idx], encoding="utf-8"))

        image = Image.open(
            self.img_dir / f"{ann['image_id']}.png"
        ).convert("RGB")

        label_ids = [label2id[l] for l in ann["labels"]]

        encoding = self.processor(
            image,
            ann["tokens"],
            boxes=ann["bboxes"],
            word_labels=label_ids,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        return {k: v.squeeze(0) for k, v in encoding.items()}


In [11]:
dataset = FoodLabelDataset(
    ann_dir=ANN_DIR,
    img_dir=IMG_DIR,
    processor=processor
)

In [12]:
sample = dataset[0]
for k, v in sample.items():
    print(k, v.shape)

input_ids torch.Size([512])
attention_mask torch.Size([512])
bbox torch.Size([512, 4])
labels torch.Size([512])
pixel_values torch.Size([3, 224, 224])


In [13]:
dataloader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True
)

In [14]:
from labels import label2id, id2label

model = LayoutLMv3ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

Some weights of LayoutLMv3ForTokenClassification were not initialized from the model checkpoint at microsoft/layoutlmv3-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

LayoutLMv3ForTokenClassification(
  (layoutlmv3): LayoutLMv3Model(
    (embeddings): LayoutLMv3TextEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (x_position_embeddings): Embedding(1024, 128)
      (y_position_embeddings): Embedding(1024, 128)
      (h_position_embeddings): Embedding(1024, 128)
      (w_position_embeddings): Embedding(1024, 128)
    )
    (patch_embed): LayoutLMv3PatchEmbeddings(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (norm): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
    (encoder): LayoutLMv3Encoder

In [16]:
batch = next(iter(dataloader))
batch = {k: v.to(device) for k, v in batch.items()}

outputs = model(**batch)
print(outputs.loss)

c:\Projects\Research_Project\env\Lib\site-packages\transformers\modeling_utils.py:1621: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


tensor(3.2114, grad_fn=<NllLossBackward0>)


In [17]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

In [19]:
from torch.utils.data import DataLoader, random_split

# ---------------- SPLIT RATIO ----------------
TRAIN_RATIO = 0.8

dataset_size = len(dataset)
train_size = int(TRAIN_RATIO * dataset_size)
val_size = dataset_size - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size]
)

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")


Train samples: 111
Validation samples: 28


In [20]:
BATCH_SIZE = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


In [21]:
from tqdm import tqdm

EPOCHS = 3

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    # ================= TRAIN =================
    model.train()
    train_loss = 0.0

    for batch in tqdm(train_loader, desc="Training"):
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # ================= VALIDATION =================
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)
            loss = outputs.loss

            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    # ================= LOG =================
    print(
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f}"
    )



Epoch 1/3


Validation: 100%|██████████| 14/14 [00:20<00:00,  1.43s/it]


Train Loss: 1.1019 | Val Loss: 0.7812

Epoch 2/3


Validation: 100%|██████████| 14/14 [00:20<00:00,  1.47s/it]


Train Loss: 0.6202 | Val Loss: 0.4611

Epoch 3/3


Validation: 100%|██████████| 14/14 [00:20<00:00,  1.47s/it]

Train Loss: 0.3717 | Val Loss: 0.2873


In [22]:
import torch
import numpy as np
from sklearn.metrics import precision_recall_fscore_support

In [24]:
def evaluate_token_classification(model, dataloader, device, id2label):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)
            logits = outputs.logits  # (B, T, C)

            predictions = torch.argmax(logits, dim=-1)
            labels = batch["labels"]

            for pred_seq, label_seq in zip(predictions, labels):
                for p, l in zip(pred_seq, label_seq):
                    if l.item() == -100:
                        continue  # ignore padding

                    all_preds.append(p.item())
                    all_labels.append(l.item())

    # Convert label IDs → label names (optional but recommended)
    all_preds = [id2label[p] for p in all_preds]
    all_labels = [id2label[l] for l in all_labels]

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average="micro",
        zero_division=0
    )

    return precision, recall, f1


In [21]:
precision, recall, f1, _ = precision_recall_fscore_support(
    all_labels,
    all_preds,
    average="weighted",   # good for imbalanced labels
    zero_division=0
)

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")


Precision: 0.3794
Recall:    0.5898
F1-score:  0.4531


In [38]:
SAVE_DIR = "D:\Y4 Research\Models"
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

[]

In [39]:
def build_json(tokens, labels):
    ingredients = []
    nutrition_facts = {}
    claims = []
    serving_size = None
    calories = None

    current_entity = []
    current_type = None
    current_nutrient_name = None

    def flush_entity():
        nonlocal current_entity, current_type, current_nutrient_name
        text = " ".join(current_entity).strip()

        if not text:
            return

        if current_type == "INGREDIENT":
            ingredients.append(text)

        elif current_type == "NUTRIENT_NAME":
            current_nutrient_name = text.lower()

        elif current_type == "NUTRIENT_VALUE" and current_nutrient_name:
            nutrition_facts[current_nutrient_name] = text
            current_nutrient_name = None

        elif current_type == "SERVING_SIZE":
            nonlocal_serving[0] = text

        elif current_type == "CALORIES":
            nonlocal_calories[0] = text

        elif current_type == "CLAIMS":
            claims.append(text)

        current_entity = []
        current_type = None
    nonlocal_serving = [None]
    nonlocal_calories = [None]

    for token, label in zip(tokens, labels):

        if label == "O":
            flush_entity()
            continue

        tag, entity_type = label.split("-", 1)

        if tag == "S":
            current_entity = [token]
            current_type = entity_type
            flush_entity()

        elif tag == "B":
            flush_entity()
            current_entity = [token]
            current_type = entity_type

        elif tag == "I":
            if current_type == entity_type:
                current_entity.append(token)

        elif tag == "E":
            if current_type == entity_type:
                current_entity.append(token)
                flush_entity()

    flush_entity()

    return {
        "ingredients": ingredients,
        "nutrition_facts": nutrition_facts,
        "serving_size": nonlocal_serving[0],
        "calories": nonlocal_calories[0],
        "claims": claims
    }


In [41]:
import torch
from transformers import LayoutLMv3ForTokenClassification, LayoutLMv3Processor

MODEL_DIR = "D:\Y4 Research\Models"  # where you saved it

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LayoutLMv3ForTokenClassification.from_pretrained(MODEL_DIR)
processor = LayoutLMv3Processor.from_pretrained(MODEL_DIR)

model.to(device)
model.eval()


LayoutLMv3ForTokenClassification(
  (layoutlmv3): LayoutLMv3Model(
    (embeddings): LayoutLMv3TextEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (x_position_embeddings): Embedding(1024, 128)
      (y_position_embeddings): Embedding(1024, 128)
      (h_position_embeddings): Embedding(1024, 128)
      (w_position_embeddings): Embedding(1024, 128)
    )
    (patch_embed): LayoutLMv3PatchEmbeddings(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (norm): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
    (encoder): LayoutLMv3Encoder

In [42]:
sample = dataset[0]   # any index

In [43]:
inputs = {
    k: v.unsqueeze(0).to(device)
    for k, v in sample.items()
    if k != "labels"   # ❗ do not pass labels during inference
}

In [44]:
with torch.no_grad():
    outputs = model(**inputs)

In [45]:
import torch

pred_ids = torch.argmax(outputs.logits, dim=-1).squeeze(0)

pred_labels = [id2label[i.item()] for i in pred_ids]

In [46]:
tokens = json.load(open(dataset.files[0], encoding="utf-8"))["tokens"]

In [47]:
structured_json = build_json(tokens, pred_labels)

In [48]:
import json
print(json.dumps(structured_json, indent=2))

{
  "ingredients": [
    "Size",
    "Magnesium",
    "Oxide)",
    "co",
    "=",
    "o",
    "GNC",
    "supplies",
    "calcium,",
    "and",
    "_",
    "from",
    "fossilized",
    "Okinawan",
    "our",
    "coral",
    "friendly.",
    "This",
    "protects",
    "reefs",
    "pe:",
    "to",
    "CHILDREN.",
    "Store"
  ],
  "nutrition_facts": {},
  "serving_size": null,
  "calories": null,
  "claims": [
    "Two",
    "USA",
    "Amount",
    "Serving",
    "1100",
    "mg)",
    "40%.",
    "in"
  ]
}


In [35]:
import pickle